# 🎬 Content-Based Anime Recommendation: Shared Protocol

This notebook implements content-based recommendation using the **same evaluation protocol as Lijie's model-based pipeline**.

## What is predicted?

The model does not predict an exact rating or the literal next watch. For each user, it ranks a fixed set of 100 anime:

- 1 held-out anime that the user rated at least 7
- 99 warm anime the user has never rated

The primary metric is **NDCG@10**. Hit Rate@10 is secondary.

## Experiment boundary

- Training profile: training positives only
- Validation: choose one of C1–C4
- Test: score only the selected configuration once
- Existing `content_based.ipynb`: separate full-catalog diagnostic, not numerically comparable

## 🧭 Pipeline

```text
ratings + anime metadata
          ↓
exact shared 5-core and leave-two-out protocol
          ↓
C1–C4 anime vectors
          ↓
rating-weighted user profiles from train positives
          ↓
fixed validation candidates
          ↓
select highest validation NDCG@10
          ↓
fixed test candidates, evaluated once
```

The final anime vectors are concatenated feature blocks followed by one L2 normalization. There are no manually tuned block weights.

In [4]:
from __future__ import annotations

import html
import os
import re
import sys
from pathlib import Path

import numpy as np
import pandas as pd
from scipy import sparse
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import OneHotEncoder, normalize

SEED = 42
POSITIVE_THRESHOLD = 7
CORE_SIZE = 5
NEGATIVE_COUNT = 99
BATCH_SIZE = 250_000

# Optional: set this to a small integer for a quick development run.
# Leave unset for the official 60,384-user experiment.
EVAL_USER_LIMIT = int(os.environ["CONTENT_EVAL_USER_LIMIT"]) if os.getenv("CONTENT_EVAL_USER_LIMIT") else None

pd.set_option("display.max_colwidth", 80)
pd.set_option("display.max_columns", 30)

def find_repo_root(start: Path = Path.cwd()) -> Path:
    for path in (start, *start.parents):
        if (path / "src" / "data.py").exists():
            return path
    raise FileNotFoundError("Run this notebook inside the dam repository.")

REPO_ROOT = find_repo_root()
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from src.data import (
    assert_split_integrity,
    build_observed_sets,
    iterative_positive_k_core,
    leave_two_out_split,
    load_anime_ratings,
    positive_interactions,
    remove_ambiguous_pairs,
    sample_evaluation_candidates,
)
from src.evaluate import per_user_metrics, rank_candidates

print("Repository:", REPO_ROOT)
print("Evaluation mode:", "FULL" if EVAL_USER_LIMIT is None else f"SMOKE ({EVAL_USER_LIMIT:,} users)")

Repository: /home/hesiyuan/Desktop/projects/dam
Evaluation mode: FULL


## 1️⃣ Load the raw tables

Set `ANIME_DATASET_DIR` to a folder containing `anime.csv` and `rating.csv`. If it is unset, the notebook downloads the Kaggle dataset.

The raw anime table is intentional. The cleaned table omits anime 33242 because its genre is missing, but the shared candidate universe must retain all 7,223 warm items.

In [5]:
def resolve_dataset_dir() -> Path:
    configured = os.getenv("ANIME_DATASET_DIR")
    if configured:
        path = Path(configured).expanduser().resolve()
    else:
        try:
            import kagglehub
        except ImportError as exc:
            raise ImportError(
                "Install kagglehub or set ANIME_DATASET_DIR to the downloaded dataset folder."
            ) from exc
        path = Path(
            kagglehub.dataset_download("CooperUnion/anime-recommendations-database")
        )
    required = [path / "anime.csv", path / "rating.csv"]
    missing = [str(file) for file in required if not file.exists()]
    if missing:
        raise FileNotFoundError(f"Missing dataset files: {missing}")
    return path

DATASET_DIR = resolve_dataset_dir()
anime_raw = pd.read_csv(DATASET_DIR / "anime.csv")
ratings_raw = load_anime_ratings(DATASET_DIR / "rating.csv")

print("Dataset:", DATASET_DIR)
display(anime_raw.head())
display(ratings_raw.head())
print(f"Anime rows: {len(anime_raw):,}")
print(f"Rating rows: {len(ratings_raw):,}")

Dataset: /home/hesiyuan/.cache/kagglehub/datasets/CooperUnion/anime-recommendations-database/versions/1


,anime_id,name,genre,type,episodes,rating,members
0,32281,Kimi no Na wa.,"Drama, Romance, School, Supernatural",Movie,1,9.37,200630
1,5114,Fullmetal Alchemist: Brotherhood,"Action, Adventure, Drama, Fantasy, Magic, Military, Shounen",TV,64,9.26,793665
2,28977,Gintama°,"Action, Comedy, Historical, Parody, Samurai, Sci-Fi, Shounen",TV,51,9.25,114262
3,9253,Steins;Gate,"Sci-Fi, Thriller",TV,24,9.17,673572
4,9969,Gintama&#039;,"Action, Comedy, Historical, Parody, Samurai, Sci-Fi, Shounen",TV,51,9.16,151266


,user_id,anime_id,rating
0,1,20,-1
1,1,24,-1
2,1,79,-1
3,1,226,-1
4,1,241,-1


Anime rows: 12,294
Rating rows: 7,813,737


## 2️⃣ Recreate Lijie's shared protocol

Important details:

- all rows from duplicated user-anime pairs are removed
- ratings 7–10 define positives
- the iterative 5-core is computed on positives
- each user contributes one validation and one test positive
- negative candidates exclude every observed rating, including low ratings and `-1`
- validation uses seed 42; test uses seed 43

In [6]:
deduplicated, ambiguous_pair_count = remove_ambiguous_pairs(ratings_raw)
positives = positive_interactions(deduplicated, POSITIVE_THRESHOLD)
core, core_stats = iterative_positive_k_core(positives, CORE_SIZE)
split = leave_two_out_split(core, SEED)

eligible_users = sorted(int(x) for x in split.train["user_id"].unique())
warm_items = sorted(int(x) for x in split.train["anime_id"].unique())
observed_by_user = build_observed_sets(deduplicated, eligible_users)

validation_candidates_full = sample_evaluation_candidates(
    split.validation,
    observed_by_user,
    warm_items,
    negative_count=NEGATIVE_COUNT,
    seed=SEED,
)
test_candidates_full = sample_evaluation_candidates(
    split.test,
    observed_by_user,
    warm_items,
    negative_count=NEGATIVE_COUNT,
    seed=SEED + 1,
)
assert_split_integrity(
    split,
    validation_candidates_full,
    test_candidates_full,
    observed_by_user,
)

protocol_summary = pd.Series({
    "ambiguous pairs removed": ambiguous_pair_count,
    "eligible users": len(eligible_users),
    "warm items": len(warm_items),
    "train positives": len(split.train),
    "validation positives": len(split.validation),
    "test positives": len(split.test),
    "validation candidates": len(validation_candidates_full),
    "test candidates": len(test_candidates_full),
    "core iterations": core_stats.iterations,
})
protocol_display = protocol_summary.to_frame("value")
protocol_display["value"] = protocol_display["value"].map(lambda value: f"{int(value):,}")
display(protocol_display)

EXPECTED = {
    "ambiguous pairs removed": 7,
    "eligible users": 60_384,
    "warm items": 7_223,
    "train positives": 5_087_394,
    "validation positives": 60_384,
    "test positives": 60_384,
}
for key, expected in EXPECTED.items():
    actual = int(protocol_summary[key])
    assert actual == expected, f"{key}: expected {expected:,}, got {actual:,}"

print("✅ Shared protocol counts and integrity checks match.")

,value
ambiguous pairs removed,7
eligible users,"60,384"
warm items,"7,223"
train positives,"5,087,394"
validation positives,"60,384"
test positives,"60,384"
validation candidates,"6,038,400"
test candidates,"6,038,400"
core iterations,1


✅ Shared protocol counts and integrity checks match.


In [7]:
# A deterministic subset changes only runtime, not data preparation or candidate sampling.
if EVAL_USER_LIMIT is None:
    evaluation_users = eligible_users
else:
    if not 1 <= EVAL_USER_LIMIT <= len(eligible_users):
        raise ValueError("CONTENT_EVAL_USER_LIMIT is outside the eligible-user range.")
    evaluation_users = eligible_users[:EVAL_USER_LIMIT]

validation_candidates = validation_candidates_full.loc[
    validation_candidates_full["user_id"].isin(evaluation_users)
].reset_index(drop=True)
test_candidates = test_candidates_full.loc[
    test_candidates_full["user_id"].isin(evaluation_users)
].reset_index(drop=True)

assert len(validation_candidates) == len(evaluation_users) * 100
assert len(test_candidates) == len(evaluation_users) * 100
print(f"Scoring {len(evaluation_users):,} users and {len(validation_candidates):,} candidates per split.")

Scoring 60,384 users and 6,038,400 candidates per split.


## 3️⃣ Join metadata without dropping a warm item

Anime 33242 has no genre. Its C1 vector is therefore all zeros. Its type, episode count, and title still contribute in C2–C4.

In [8]:
metadata = (
    anime_raw.rename(columns={"anime_id": "item_id"})
    .drop_duplicates("item_id", keep="first")
    .set_index("item_id")
    .reindex(warm_items)
    .reset_index()
)

assert metadata["item_id"].notna().all()
assert metadata["name"].notna().all(), "Every warm item must have a title."
assert metadata["type"].dropna().isin(["Movie", "Music", "ONA", "OVA", "Special", "TV"]).all()

missing_genre = metadata.loc[metadata["genre"].isna(), ["item_id", "name", "type", "episodes"]]
display(missing_genre)
assert set(missing_genre["item_id"]) == {33242}
assert len(metadata) == 7_223
print("✅ All 7,223 warm items are retained.")

,item_id,name,type,episodes
7182,33242,IS: Infinite Stratos 2 - Infinite Wedding,Special,1


✅ All 7,223 warm items are retained.


## 4️⃣ Build C1–C4 anime vectors

| Configuration | Composition | Expected size |
| --- | --- | ---: |
| C1 | genre TF-IDF | 43 |
| C2 | genre TF-IDF + type one-hot | 49 |
| C3 | genre TF-IDF + type one-hot + episode bucket | 56 |
| C4 | genre TF-IDF + title TF-IDF | data-dependent |

Each row is normalized only after its blocks are concatenated.

In [9]:
def split_genres(text: str) -> list[str]:
    if not text or pd.isna(text):
        return []
    return [token.strip() for token in str(text).split(",") if token.strip()]

def clean_title(text: str) -> str:
    decoded = html.unescape(str(text))
    return re.sub(r"\s+", " ", decoded.lower()).strip()

EPISODE_BUCKETS = ["1", "2-6", "7-13", "14-26", "27-52", "53+", "Unknown"]

def episode_bucket(value: object) -> str:
    number = pd.to_numeric(value, errors="coerce")
    if pd.isna(number):
        return "Unknown"
    number = int(number)
    if number == 1:
        return "1"
    if number <= 6:
        return "2-6"
    if number <= 13:
        return "7-13"
    if number <= 26:
        return "14-26"
    if number <= 52:
        return "27-52"
    return "53+"

genre_vectorizer = TfidfVectorizer(
    analyzer=split_genres,
    lowercase=False,
    norm=None,
    dtype=np.float32,
)
genre_matrix = genre_vectorizer.fit_transform(metadata["genre"].fillna(""))

type_encoder = OneHotEncoder(
    categories=[["Movie", "Music", "ONA", "OVA", "Special", "TV"]],
    handle_unknown="ignore",
    sparse_output=True,
    dtype=np.float32,
)
type_matrix = type_encoder.fit_transform(metadata[["type"]].fillna("Unknown"))

episode_encoder = OneHotEncoder(
    categories=[EPISODE_BUCKETS],
    handle_unknown="ignore",
    sparse_output=True,
    dtype=np.float32,
)
episode_labels = metadata["episodes"].map(episode_bucket).to_frame("episode_bucket")
episode_matrix = episode_encoder.fit_transform(episode_labels)

title_vectorizer = TfidfVectorizer(
    ngram_range=(1, 2),
    min_df=2,
    lowercase=False,
    norm=None,
    dtype=np.float32,
)
title_matrix = title_vectorizer.fit_transform(metadata["name"].map(clean_title))

raw_configurations = {
    "C1": genre_matrix,
    "C2": sparse.hstack([genre_matrix, type_matrix], format="csr"),
    "C3": sparse.hstack([genre_matrix, type_matrix, episode_matrix], format="csr"),
    "C4": sparse.hstack([genre_matrix, title_matrix], format="csr"),
}
configurations = {
    name: normalize(matrix, norm="l2", axis=1, copy=True).tocsr().astype(np.float32)
    for name, matrix in raw_configurations.items()
}

dimension_table = pd.DataFrame({
    "configuration": list(configurations),
    "dimensions": [matrix.shape[1] for matrix in configurations.values()],
    "nonzero values": [matrix.nnz for matrix in configurations.values()],
})
display(dimension_table)
assert configurations["C1"].shape == (7_223, 43)
assert configurations["C2"].shape == (7_223, 49)
assert configurations["C3"].shape == (7_223, 56)

row_norms = {
    name: np.sqrt(matrix.multiply(matrix).sum(axis=1)).A1
    for name, matrix in configurations.items()
}
missing_genre_row = warm_items.index(33242)
assert row_norms["C1"][missing_genre_row] == 0
for name, norms in row_norms.items():
    nonzero = norms > 0
    assert np.allclose(norms[nonzero], 1.0, atol=1e-5)
print("✅ Dimensions and final L2 norms are valid.")

,configuration,dimensions,nonzero values
0,C1,43,24894
1,C2,49,32117
2,C3,56,39340
3,C4,6721,60894


✅ Dimensions and final L2 norms are valid.


## 5️⃣ Build rating-weighted user profiles

Only training positives contribute.

```text
weight = rating - 6

7 → 1, 8 → 2, 9 → 3, 10 → 4
```

Example: a rating-10 anime contributes four times as much as a rating-7 anime. The weighted sum is then L2-normalized.

In [10]:
user_to_index = {user_id: index for index, user_id in enumerate(eligible_users)}
item_to_index = {item_id: index for index, item_id in enumerate(warm_items)}

train_user_index = split.train["user_id"].map(user_to_index).to_numpy(dtype=np.int32)
train_item_index = split.train["anime_id"].map(item_to_index).to_numpy(dtype=np.int32)
train_weights = (split.train["rating"].to_numpy(dtype=np.float32) - 6.0)

assert np.isin(train_weights, [1.0, 2.0, 3.0, 4.0]).all()

user_item_weights = sparse.csr_matrix(
    (train_weights, (train_user_index, train_item_index)),
    shape=(len(eligible_users), len(warm_items)),
    dtype=np.float32,
)

def build_user_profiles(item_features: sparse.csr_matrix) -> sparse.csr_matrix:
    profiles = (user_item_weights @ item_features).tocsr()
    return normalize(profiles, norm="l2", axis=1, copy=False).tocsr()

def weighted_profile_example() -> pd.DataFrame:
    toy_items = sparse.csr_matrix(
        np.array([[1.0, 0.0], [0.0, 1.0], [1.0, 1.0]], dtype=np.float32)
    )
    toy_weights = sparse.csr_matrix(
        np.array([[4.0, 3.0, 1.0]], dtype=np.float32)
    )
    normalized_items = normalize(toy_items, axis=1)
    actual = normalize(toy_weights @ normalized_items, axis=1)
    unnormalized_expected = np.array([[4 + 1 / np.sqrt(2), 3 + 1 / np.sqrt(2)]])
    expected = unnormalized_expected / np.linalg.norm(unnormalized_expected)
    actual_array = actual.toarray() if sparse.issparse(actual) else np.asarray(actual)
    assert np.allclose(expected, actual_array)
    return pd.DataFrame(actual_array, columns=["feature A", "feature B"])

display(weighted_profile_example())
print("✅ Sparse weighted-profile construction passed the hand-calculated invariant check.")

,feature A,feature B
0,0.785615,0.618715


✅ Sparse weighted-profile construction passed the hand-calculated invariant check.


## 6️⃣ Score candidates and verify evaluator parity

Cosine similarity becomes a dot product because item vectors and user profiles are L2-normalized.

The popularity check is a protocol gate. On the full experiment it must reproduce Lijie's validation result.

In [ ]:
def score_candidates(
    candidates: pd.DataFrame,
    profiles: sparse.csr_matrix,
    item_features: sparse.csr_matrix,
    batch_size: int = BATCH_SIZE,
) -> pd.DataFrame:
    user_indices = candidates["user_id"].map(user_to_index)
    item_indices = candidates["item_id"].map(item_to_index)
    if user_indices.isna().any() or item_indices.isna().any():
        raise ValueError("Candidates contain a user or item outside the warm mappings.")

    user_indices = user_indices.to_numpy(dtype=np.int32)
    item_indices = item_indices.to_numpy(dtype=np.int32)
    scores = np.empty(len(candidates), dtype=np.float32)

    for start in range(0, len(candidates), batch_size):
        stop = min(start + batch_size, len(candidates))
        user_batch = profiles[user_indices[start:stop]]
        item_batch = item_features[item_indices[start:stop]]
        scores[start:stop] = np.asarray(
            user_batch.multiply(item_batch).sum(axis=1)
        ).ravel()

    predictions = candidates.copy()
    predictions["score"] = scores
    return predictions

def evaluate_predictions(predictions: pd.DataFrame) -> tuple[dict[str, float], pd.DataFrame]:
    user_metrics = per_user_metrics(predictions, k=10)
    metrics = {
        "ndcg@10": float(user_metrics["ndcg@10"].mean()),
        "hit_rate@10": float(user_metrics["hit_rate@10"].mean()),
        "user_count": int(len(user_metrics)),
    }
    return metrics, user_metrics

popularity = split.train.groupby("anime_id")["user_id"].nunique()
popularity_predictions = validation_candidates.copy()
popularity_predictions["score"] = (
    popularity_predictions["item_id"].map(popularity).fillna(0).astype(float)
)
popularity_metrics, _ = evaluate_predictions(popularity_predictions)
display(pd.Series(popularity_metrics).to_frame("validation"))

if EVAL_USER_LIMIT is None:
    assert np.isclose(popularity_metrics["ndcg@10"], 0.5056794956640929)
    print("✅ Exact full-protocol popularity parity reproduced.")
else:
    print("ℹ️ Subset metric shown. Run without CONTENT_EVAL_USER_LIMIT for the parity assertion.")

,validation
ndcg@10,0.505679
hit_rate@10,0.770684
user_count,60384.000000


✅ Exact full-protocol popularity parity reproduced.


## 7️⃣ Validation model selection

C1–C4 are compared only on the fixed validation candidates. The winner is selected by NDCG@10. Test candidates are not scored in this section.

In [ ]:
validation_rows = []
validation_user_ranks = {}
c4_validation_top10 = None

for configuration_name, item_features in configurations.items():
    print(f"Evaluating {configuration_name} ...")
    profiles = build_user_profiles(item_features)
    predictions = score_candidates(validation_candidates, profiles, item_features)
    metrics, user_metrics = evaluate_predictions(predictions)
    validation_rows.append({"configuration": configuration_name, **metrics})
    validation_user_ranks[configuration_name] = user_metrics[
        ["user_id", "rank", "ndcg@10", "hit_rate@10"]
    ].copy()
    if configuration_name == "C4":
        c4_ranked = rank_candidates(predictions)
        c4_validation_top10 = c4_ranked.loc[c4_ranked["rank"].le(10)].copy()
        del c4_ranked
    del profiles, predictions directly insert the raw episode count.

validation_results = (
    pd.DataFrame(validation_rows)
    .sort_values(["ndcg@10", "configuration"], ascending=[False, True])
    .reset_index(drop=True)
)
selected_configuration = str(validation_results.loc[0, "configuration"])

display(validation_results.round({"ndcg@10": 6, "hit_rate@10": 6}))
print("Selected by validation NDCG@10:", selected_configuration)

Evaluating C1 ...
Evaluating C2 ...
Evaluating C3 ...
Evaluating C4 ...


,configuration,ndcg@10,hit_rate@10,user_count
0,C4,0.261212,0.470356,60384
1,C3,0.242140,0.427232,60384
2,C2,0.235430,0.417809,60384
3,C1,0.218531,0.393349,60384


Selected by validation NDCG@10: C4


## 8️⃣ Final test evaluation

This is the first and only model-scoring call on the fixed test candidates. Do not use this result to revise C1–C4.

In [13]:
selected_item_features = configurations[selected_configuration]
selected_profiles = build_user_profiles(selected_item_features)
test_predictions = score_candidates(
    test_candidates,
    selected_profiles,
    selected_item_features,
)
test_metrics, test_user_metrics = evaluate_predictions(test_predictions)

final_result = pd.DataFrame([{
    "configuration": selected_configuration,
    "protocol": "1 positive + 99 fixed unseen negatives",
    **test_metrics,
    "evaluation_mode": "full" if EVAL_USER_LIMIT is None else f"smoke:{EVAL_USER_LIMIT}",
}])
display(final_result.round({"ndcg@10": 6, "hit_rate@10": 6}))

,configuration,protocol,ndcg@10,hit_rate@10,user_count,evaluation_mode
0,C4,1 positive + 99 fixed unseen negatives,0.259747,0.469114,60384,full


## 9️⃣ Intuitive success and failure cases

For a user's held-out anime, a training anime's contribution is approximated by:

```text
(rating - 6) × cosine(training anime, held-out anime)
```

The following cells show:

- a rank-1 success
- a boundary hit near rank 10
- a failure just outside the top 10
- a case where C4 title features substantially change the held-out rank
- a heuristic franchise-saturation example among C4's top 10

In [14]:
ranked_test = rank_candidates(test_predictions)
positive_test = ranked_test.loc[ranked_test["label"].eq(1), ["user_id", "item_id", "rank"]]

def first_user(frame: pd.DataFrame, sort_column: str = "rank", ascending: bool = True):
    if frame.empty:
        return None
    return int(frame.sort_values(sort_column, ascending=ascending).iloc[0]["user_id"])

case_users = {
    "rank-1 success": first_user(positive_test.loc[positive_test["rank"].eq(1)]),
    "top-10 boundary": first_user(
        positive_test.loc[positive_test["rank"].le(10)], ascending=False
    ),
    "failure outside top 10": first_user(
        positive_test.loc[positive_test["rank"].gt(10)]
    ),
}
display(pd.Series(case_users, name="user_id").to_frame())

,user_id
rank-1 success,73487
top-10 boundary,14314
failure outside top 10,15696


In [15]:
metadata_by_item = metadata.set_index("item_id")

def explain_user(user_id: int, top_n: int = 10) -> tuple[pd.DataFrame, pd.DataFrame]:
    candidate_top = (
        ranked_test.loc[ranked_test["user_id"].eq(user_id)]
        .head(top_n)
        .merge(metadata[["item_id", "name", "genre", "type", "episodes"]], on="item_id", how="left")
    )
    positive_item = int(
        ranked_test.loc[
            ranked_test["user_id"].eq(user_id) & ranked_test["label"].eq(1), "item_id"
        ].iloc[0]
    )
    target_index = item_to_index[positive_item]
    history = split.train.loc[split.train["user_id"].eq(user_id)].copy()
    history_indices = history["anime_id"].map(item_to_index).to_numpy(dtype=np.int32)
    similarities = np.asarray(
        selected_item_features[history_indices]
        .multiply(selected_item_features[target_index])
        .sum(axis=1)
    ).ravel()
    history["contribution"] = (history["rating"].to_numpy() - 6) * similarities
    contributors = (
        history.nlargest(5, "contribution")
        .rename(columns={"anime_id": "item_id"})
        .merge(metadata[["item_id", "name", "genre", "type", "episodes"]], on="item_id", how="left")
    )
    return candidate_top, contributors

for label, user_id in case_users.items():
    if user_id is None:
        print(f"No {label} exists in this evaluation subset.")
        continue
    print(f"\n{label.upper()} | user {user_id}")
    top_candidates, contributors = explain_user(user_id)
    display(top_candidates[["rank", "label", "score", "item_id", "name", "genre", "type", "episodes"]])
    print("Strongest training contributors to the held-out anime:")
    display(contributors[["rating", "contribution", "item_id", "name", "genre", "type", "episodes"]])


RANK-1 SUCCESS | user 73487


,rank,label,score,item_id,name,genre,type,episodes
0,1,1,0.309630,384,Gantz,"Action, Drama, Horror, Psychological, Sci-Fi, Supernatural",TV,13
1,2,0,0.297451,267,Gungrave,"Action, Drama, Sci-Fi, Seinen, Super Power",TV,26
2,3,0,0.232070,21027,Saibi,"Drama, Psychological, Thriller",Movie,1
3,4,0,0.219420,3060,Dallos,"Action, Mecha, Sci-Fi",OVA,4
4,5,0,0.192399,1832,Meiou Project Zeorymer,"Action, Drama, Mecha, Psychological, Sci-Fi",OVA,4
5,6,0,0.185544,3254,Luv Wave,"Action, Drama, Hentai, Horror, Mystery, Police, Romance, Sci-Fi",OVA,3
6,7,0,0.131317,6682,11eyes,"Action, Ecchi, Super Power, Supernatural",TV,12
7,8,0,0.126633,619,Ninja Ryuukenden,"Adventure, Demons, Drama, Sci-Fi, Shounen, Supernatural, Thriller",OVA,1
8,9,0,0.114064,3976,Mamahaha,"Drama, Hentai, Romance",OVA,2
9,10,0,0.113823,1702,Yoru ga Kuru!,"Demons, Hentai, Horror, Super Power, Supernatural",OVA,4


Strongest training contributors to the held-out anime:


,rating,contribution,item_id,name,genre,type,episodes
0,10,1.159024,226,Elfen Lied,"Action, Drama, Horror, Psychological, Romance, Seinen, Supernatural",TV,13
1,7,0.809499,395,Gantz 2nd Stage,"Action, Drama, Horror, Psychological, Sci-Fi, Supernatural",TV,13
2,8,0.298587,1535,Death Note,"Mystery, Police, Psychological, Supernatural, Thriller",TV,37
3,10,0.245277,2904,Code Geass: Hangyaku no Lelouch R2,"Action, Drama, Mecha, Military, Sci-Fi, Super Power",TV,25
4,9,0.129726,1575,Code Geass: Hangyaku no Lelouch,"Action, Mecha, Military, School, Sci-Fi, Super Power",TV,25



TOP-10 BOUNDARY | user 14314


...

,rating,contribution,item_id,name,genre,type,episodes
0,8,1.601671,534,Slayers,"Adventure, Comedy, Demons, Fantasy, Magic",TV,26
1,10,0.424218,20159,Pokemon: The Origin,"Action, Adventure, Comedy, Fantasy, Kids",Special,4
2,9,0.376120,17265,Log Horizon,"Action, Adventure, Fantasy, Game, Magic, Shounen",TV,25
3,8,0.323377,1050,Bobobo-bo Bo-bobo,"Adventure, Comedy, Sci-Fi, Shounen",TV,76
4,9,0.311903,5114,Fullmetal Alchemist: Brotherhood,"Action, Adventure, Drama, Fantasy, Magic, Military, Shounen",TV,64



FAILURE OUTSIDE TOP 10 | user 15696


,rank,label,score,item_id,name,genre,type,episodes
0,1,0,0.413303,2439,Hooligan,Hentai,OVA,2
1,2,0,0.413303,9970,Floating Material,Hentai,OVA,2
2,3,0,0.377675,1076,Wizardry,"Action, Adventure, Fantasy",OVA,1
3,4,0,0.341308,599,Yuugen Kaisha,"Action, Adventure, Comedy, Magic, Super Power, Supernatural",OVA,4
4,5,0,0.270961,1558,Yarima Queen,"Comedy, Demons, Fantasy, Hentai, Magic, Parody",OVA,1
5,6,0,0.267374,1722,Nagasarete Airantou,"Comedy, Ecchi, Fantasy, Harem, Romance",TV,26
6,7,0,0.218173,15751,Senyuu.,"Action, Comedy, Fantasy, Parody, Shounen",TV,13
7,8,0,0.209530,2287,Fighting Foodons,"Adventure, Comedy, Fantasy, Game, Shounen",TV,26
8,9,0,0.199151,2315,Injuu Alien,"Action, Adventure, Hentai, Horror, Sci-Fi",OVA,1
9,10,0,0.185383,6590,Makai Kishi Ingrid,"Demons, Fantasy, Hentai, Martial Arts",OVA,4


Strongest training contributors to the held-out anime:


,rating,contribution,item_id,name,genre,type,episodes
0,10,1.734348,3537,Cleavage,Hentai,OVA,2
1,8,0.661597,2372,Enbo,"Drama, Hentai",OVA,6
2,10,0.549662,2432,Discipline,"Hentai, School",OVA,6
3,7,0.433587,3896,Inbo,Hentai,OVA,3
4,9,0.394699,4358,Aniyome wa Ijippari,Hentai,OVA,2


In [16]:
# Compare held-out ranks across configurations.
rank_comparison = None
for name, frame in validation_user_ranks.items():
    current = frame[["user_id", "rank"]].rename(columns={"rank": f"{name}_rank"})
    rank_comparison = current if rank_comparison is None else rank_comparison.merge(current, on="user_id")

rank_comparison["C4_minus_C1_improvement"] = (
    rank_comparison["C1_rank"] - rank_comparison["C4_rank"]
)
display(rank_comparison.nlargest(10, "C4_minus_C1_improvement"))

def franchise_key(title: str) -> str:
    tokens = re.findall(r"[a-z0-9]+", clean_title(title))
    stop = {"season", "movie", "special", "ova", "tv", "the", "part"}
    tokens = [token for token in tokens if token not in stop and not token.isdigit()]
    return " ".join(tokens[:2])

c4_top10 = c4_validation_top10.merge(
    metadata[["item_id", "name"]], on="item_id", how="left"
)
c4_top10["franchise_key"] = c4_top10["name"].map(franchise_key)
franchise_counts = (
    c4_top10.groupby(["user_id", "franchise_key"]).size()
    .rename("same_key_count")
    .reset_index()
)
franchise_case = franchise_counts.sort_values(
    ["same_key_count", "user_id"], ascending=[False, True]
).iloc[0]
print(
    "C4 franchise-saturation heuristic:",
    f"user={int(franchise_case.user_id)},",
    f"key='{franchise_case.franchise_key}',",
    f"top-10 count={int(franchise_case.same_key_count)}",
)
display(
    c4_top10.loc[c4_top10["user_id"].eq(int(franchise_case.user_id)),
                 ["rank", "score", "label", "item_id", "name", "franchise_key"]]
)

,user_id,C1_rank,C2_rank,C3_rank,C4_rank,C4_minus_C1_improvement
32745,39893,100,70,63,1,99
57181,69544,99,97,92,1,98
7576,9208,98,98,98,1,97
13136,16044,93,97,96,3,90
56270,68410,90,91,91,1,89
13571,16580,94,89,91,6,88
26852,32629,89,90,91,1,88
17194,20967,94,96,89,8,86
17096,20848,89,84,82,6,83
18506,22552,84,85,85,1,83


C4 franchise-saturation heuristic: user=51773, key='dragon ball', top-10 count=5


,rank,score,label,item_id,name,franchise_key
425470,1,0.438961,0,813,Dragon Ball Z,dragon ball
425471,2,0.378664,0,22695,Dragon Ball Z: Summer Vacation Special,dragon ball
425472,3,0.352456,0,33740,Katekyo Hitman Reborn! x ēlDLIVE Special,katekyo hitman
425473,4,0.291118,0,891,Dragon Ball Movie 2: Majinjou no Nemuri Hime,dragon ball
425474,5,0.276131,0,893,Dragon Ball Movie 4: Saikyou e no Michi,dragon ball
425475,6,0.268346,0,2617,Ghiblies,ghiblies
425476,7,0.232355,0,18095,Nourin,nourin
425477,8,0.207909,1,898,Dragon Ball Z Movie 05: Tobikkiri no Saikyou tai Saikyou,dragon ball
425478,9,0.202386,0,2171,Detective Conan Movie 11: Jolly Roger in the Deep Azure,detective conan
425479,10,0.178211,0,3574,Switch,switch


## 🔟 Handoff record

Record the completed run below.

- Dataset: CooperUnion Anime Recommendations Database, Kaggle version 1
- Split: seeded random leave-two-out with warm-item repair
- Positive threshold: 7
- Positive core: iterative 5-core
- Validation candidates: seed 42
- Test candidates: seed 43
- Profile confidence: rating − 6
- Selection metric: validation NDCG@10
- Test policy: selected configuration evaluated once
- Tie rule: descending score, then ascending anime ID

### Limitations

- Content similarity cannot learn broad crowd preference as directly as collaborative models.
- C1 gives anime 33242 a zero vector because its genre is missing.
- C4 may over-recommend titles from the same franchise.
- Sampled-candidate metrics are useful for shared comparison, but they are easier than ranking the entire catalog.
- A cold-start user still requires at least one liked anime to form a content profile.